# DRUGseqPy — Notebook 2: Screen-Level Analysis

Continuing from `01_QC_Workflow.ipynb`, this notebook covers:
1. Batch correction (Harmony / DMSO-regression)
2. PCA, UMAP, and DMSO-anchored perturbation scoring
3. Parallelised differential expression for all compounds
4. Volcano, MA, and gene-count plots
5. Screen overview and logFC heatmap
6. Compound fingerprints and similarity network
7. Compound-level UMAP (DE-signature aggregation)
8. GSEA and connectivity scoring

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import drugseqpy as ds
from drugseqpy.utils import make_dummy_screen

sc.settings.verbosity = 1
plt.rcParams['figure.dpi'] = 100
print(f'drugseqpy v{ds.__version__}')

## 0. Load filtered data from Notebook 1

In [ ]:
# Load from Notebook 1 output:
# adata = ad.read_h5ad('dsd_qc_filtered.h5ad')
# dsd = ds.DrugSeqData(adata)

# Or rebuild from scratch for this tutorial:
counts, obs = make_dummy_screen(
    n_genes=500, n_plates=2,
    n_dmso_per_plate=10, n_compounds=6, n_reps=4, seed=42
)
dsd = ds.create_drugseq_object(counts=counts, obs=obs)
ds.compute_qc_metrics(dsd, inplace=True)
ds.compute_plate_qc(dsd, inplace=True)
ds.normalize_counts(dsd, method='limma_voom', inplace=True)
dsd = ds.filter_genes(dsd, min_count=5, min_samples=2, group_aware=True)
print(dsd)

## 1. PCA

Runs scanpy's PCA on normalized counts, selecting highly variable genes first.

In [ ]:
ds.run_pca(dsd, n_pcs=50, n_variable_genes=300, inplace=True)

In [ ]:
fig = ds.plot_pc_elbow(dsd, n_pcs=30)
plt.show()

In [ ]:
fig = ds.plot_embedding(dsd, reduction='X_pca', dims=(0, 1),
                         color_by='sample_type')
plt.show()

## 2. Batch correction

Harmony corrects PCA embeddings across plates without modifying normalized counts.

In [ ]:
# Harmony integration (corrects X_pca → X_pca_harmony)
ds.correct_batch(dsd, batch_col='plate_id', method='harmony', n_pcs=30, inplace=True)

# Check: post-Harmony PCA should cluster by compound, not plate
fig = ds.plot_embedding(dsd, reduction='X_pca_harmony', dims=(0, 1),
                         color_by='plate_id')
plt.title('Post-Harmony: coloured by plate (should be mixed)')
plt.show()

In [ ]:
# Alternative: DMSO-regression (Drug-seq native; corrects adata.X directly)
# ds.correct_batch(dsd, batch_col='plate_id', method='dmso_regression', inplace=True)

## 3. UMAP

In [ ]:
ds.run_umap(dsd, dims=20, n_neighbors=15, min_dist=0.3,
             use_harmony=True, inplace=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col in zip(axes, ['compound', 'plate_id']):
    ds.plot_embedding(dsd, reduction='X_umap', color_by=col, figsize=(6,5))
plt.tight_layout()
plt.show()

## 4. DMSO-anchored perturbation scoring

Fits PCA on DMSO wells only, projects all samples into that space, and
computes a z-scored distance from the DMSO centroid.  High `pert_score` → strong transcriptional response.

In [ ]:
ds.embed_dmso(dsd, n_pcs=15, n_variable_genes=300, inplace=True)

# Distribution of perturbation scores
import seaborn as sns
fig, ax = plt.subplots(figsize=(8, 3))
df_pert = dsd.obs[['compound', 'sample_type', 'pert_score']]
order = df_pert.groupby('compound')['pert_score'].median().sort_values().index
sns.boxplot(data=df_pert, x='compound', y='pert_score', order=order, ax=ax,
            palette='Blues', linewidth=0.8)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.7)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
ax.set_title('Perturbation score vs DMSO centroid')
plt.tight_layout()
plt.show()

## 5. Differential expression

All compounds vs DMSO, parallelised via `n_jobs`.
- **pydeseq2** (default): Python port of DESeq2 — best for count data
- **ols_voom**: OLS on voom log-CPM — equivalent to limma-voom in R
- **t_test**: fast screening

In [ ]:
ds.compute_multi_de(
    dsd,
    reference='DMSO',
    method='ols_voom',   # or 'pydeseq2'
    within_plate=True,
    fdr_threshold=0.05,
    lfc_threshold=0.5,
    n_jobs=1,            # increase for large screens
    inplace=True,
)

In [ ]:
# Summary table: one row per compound
de_sum = ds.summarise_de(dsd)
de_sum.sort_values('n_sig_total', ascending=False)

## 6. Volcano and MA plots

In [ ]:
# Plot for each compound in a grid
compounds = list(dsd.adata.uns['de_results'].keys())
n_cpds = len(compounds)
ncol = 3
nrow = int(np.ceil(n_cpds / ncol))

fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*5, nrow*4))
for ax, cmpd in zip(axes.ravel(), compounds):
    df = dsd.adata.uns['de_results'][cmpd].copy()
    df['-log10_padj'] = -np.log10(df['padj'].clip(1e-300))
    sig = df['significant'].fillna(False)
    ax.scatter(df.loc[~sig,'logFC'], df.loc[~sig,'-log10_padj'],
               s=6, alpha=0.4, color='#AAAAAA', edgecolors='none')
    ax.scatter(df.loc[sig & (df.logFC>0),'logFC'], df.loc[sig & (df.logFC>0),'-log10_padj'],
               s=8, alpha=0.8, color='#C0392B', edgecolors='none')
    ax.scatter(df.loc[sig & (df.logFC<0),'logFC'], df.loc[sig & (df.logFC<0),'-log10_padj'],
               s=8, alpha=0.8, color='#2980B9', edgecolors='none')
    ax.axvline(-0.5, color='grey', linestyle='--', lw=0.5)
    ax.axvline( 0.5, color='grey', linestyle='--', lw=0.5)
    ax.axhline(-np.log10(0.05), color='grey', linestyle='--', lw=0.5)
    n_up = int((sig & (df.logFC>0)).sum())
    n_dn = int((sig & (df.logFC<0)).sum())
    ax.set_title(f'{cmpd}  ↑{n_up} ↓{n_dn}', fontsize=8)
    ax.set_xlabel('log₂FC', fontsize=7)
    ax.set_ylabel('-log₁₀(padj)', fontsize=7)

for ax in axes.ravel()[n_cpds:]:
    ax.set_visible(False)
fig.suptitle('Volcano plots: all compounds vs DMSO', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Gene-count box plots for specific genes of interest
sig_genes = (
    dsd.adata.uns['de_results'][compounds[0]]
    .query('significant == True')
    .nsmallest(6, 'padj')['gene'].tolist()
)

if sig_genes:
    fig = ds.plot_gene_counts(dsd, genes=sig_genes, group_by='compound')
    plt.show()
else:
    print('No significant genes at default thresholds (expected with synthetic data)')

## 7. Screen overview and logFC heatmap

In [ ]:
# Screen overview: perturbation score × DE count × direction
fig = ds.plot_screen_overview(
    dsd,
    fdr_threshold=0.05, lfc_threshold=0.5,
    use_pert_score=True,
    top_n_label=5,
)
plt.show()

In [ ]:
# LogFC heatmap across all compounds (macpie::plot_multi_de)
try:
    fig = ds.plot_screen_heatmap(dsd, n_top=40, fdr_threshold=1.0)
    plt.show()
except Exception as e:
    print(f'Heatmap skipped: {e}')

## 8. Compound fingerprints and similarity network

In [ ]:
# Signed binary {-1, 0, +1} fingerprint matrix
fp = ds.compute_compound_fingerprint(dsd, fdr_threshold=0.1, lfc_threshold=0.3)
print(f'Fingerprint matrix: {fp.shape[0]} genes × {fp.shape[1]} compounds')
print(f'Unique values: {sorted(fp.values.ravel())[:4]}…')

In [ ]:
# Cosine similarity between compound DE signatures
sim = ds.connectivity_score(dsd, method='cosine')
sim_df = pd.DataFrame(sim,
                       index=list(dsd.adata.uns['de_results'].keys()),
                       columns=list(dsd.adata.uns['de_results'].keys()))

import seaborn as sns
fig, ax = plt.subplots(figsize=(6, 5))
mask = np.eye(len(sim_df), dtype=bool)
sns.heatmap(sim_df, cmap='RdBu_r', center=0, ax=ax, mask=mask,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'label': 'cosine similarity', 'shrink': 0.7})
ax.set_title('Compound–compound cosine similarity')
plt.tight_layout()
plt.show()

In [ ]:
# Similarity network (requires igraph for Leiden clustering)
net = ds.compute_compound_similarity_network(dsd, method='cosine', threshold=0.0)

if net['graph'] is not None:
    import igraph as ig
    import matplotlib.pyplot as plt
    g = net['graph']
    clusters = net['clusters']
    n_clusters = clusters.nunique()
    palette = plt.cm.tab10(np.linspace(0, 1, n_clusters))
    color_map = dict(zip(sorted(clusters.unique()), palette))
    colors = [color_map[clusters[v['name']]] for v in g.vs]
    fig, ax = plt.subplots(figsize=(6, 5))
    ig.plot(g, target=ax,
            vertex_label=g.vs['name'],
            vertex_color=[list(c) for c in colors],
            edge_width=[g.es['weight'][i] * 3 for i in range(g.ecount())],
            vertex_size=0.4, layout='fr')
    ax.set_title(f'Compound network ({n_clusters} Leiden communities)')
    plt.show()
else:
    print('Install igraph for network visualization: pip install igraph')

## 9. Compound-level UMAP on DE signatures

Aggregates replicate log-fold-changes into compound-level vectors, runs PCA
then UMAP on the compound axis (macpie `aggregate_by_de` strategy).

In [ ]:
try:
    umap_df = ds.compute_compound_umap(
        dsd, n_pcs=5, n_variable_genes=200,
        n_neighbors=3, min_dist=0.3,
        use_leiden=True, resolution=0.8,
    )
    fig = ds.plot_compound_umap(umap_df, color_by='cluster', label_compounds=True)
    plt.show()
    print(umap_df)
except Exception as e:
    print(f'Compound UMAP skipped (need umap-learn): {e}')

## 10. GSEA

Requires gseapy. Uses gseapy.prerank on the per-compound DE ranking.

In [ ]:
try:
    # Using a custom gene set for reproducibility (replace with MSigDB shorthand
    # like 'MSigDB_Hallmark_2020' in real analyses)
    all_genes = dsd.var_names.tolist()
    custom_sets = {
        'GeneModule_A': all_genes[:30],
        'GeneModule_B': all_genes[30:60],
        'GeneModule_C': all_genes[60:90],
    }
    ds.run_gsea(dsd, gene_sets=custom_sets, n_perm=200, min_size=10, inplace=True)
    # Show results for first compound
    cmpd = list(dsd.adata.uns['de_results'].keys())[0]
    print(f'GSEA for {cmpd}:')
    print(dsd.adata.uns['gsea'][cmpd][['Term','NES','NOM p-val','FDR q-val']].head(10))
except Exception as e:
    print(f'GSEA skipped: {e}')

## 11. Save results

In [ ]:
dsd.adata.write_h5ad('dsd_screen_results.h5ad')
print('Saved: dsd_screen_results.h5ad')

# Export DE summary to CSV
de_sum = ds.summarise_de(dsd)
de_sum.to_csv('de_summary.csv', index=False)
print('Saved: de_summary.csv')

---
**Next →** `03_Dose_Response_Cheminformatics.ipynb`